# S1 · Data quality and frozen analytical specification

This notebook makes the final sample and timing decisions inspectable. It documents a key limitation discovered in the full extract: every valid loan has a `raisedDate`, so the data do not support a true right-censored survival analysis.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import numpy as np
import psutil
from IPython.display import display, Image

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"
AUDIT = ROOT / "audit"
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
RUN_LOG = LOGS / "run_log.txt"

def checkpoint(label, *frames, started=None):
    elapsed = time.time() - started if started is not None else 0.0
    shapes = [getattr(x, "shape", None) for x in frames]
    nulls = []
    for frame in frames:
        if hasattr(frame, "isna"):
            nulls.append(round(float(frame.isna().mean(numeric_only=False).mean()), 6))
    rss = psutil.Process().memory_info().rss / 1024**3
    line = f"{label} | shapes={shapes} | mean_null_rates={nulls} | rss_gib={rss:.3f} | elapsed_s={elapsed:.3f}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")

t0 = time.time()
print(f"Audit root: {ROOT}")
checkpoint("setup", started=t0)

Audit root: <PROJECT_ROOT>/5_最终交付包/_rebuild/MA-Hackathon-Final-2026-09-04
setup | shapes=[] | mean_null_rates=[] | rss_gib=0.113 | elapsed_s=0.000


In [2]:
t0 = time.time()
profile = pd.read_csv(OUTPUTS / "data_profile_summary.csv")
spec = pd.read_csv(OUTPUTS / "frozen_spec.csv")
quantiles = pd.read_csv(OUTPUTS / "funding_duration_quantiles_train_2016_2024.csv")
display(profile)
display(spec)
display(quantiles)
checkpoint("S1 profile and freeze", profile, spec, quantiles, started=t0)

,rows,distinct_ids,null_ids,invalid_fundraising_dates,invalid_raised_dates,negative_durations,zero_durations,min_fundraising_ts,max_fundraising_ts,min_raised_ts,max_raised_ts,sectors,activities,countries
0,1453846,1453846,0.0,0.0,0.0,6.0,0.0,2016-01-01 02:30:02+00:00,2025-12-31 20:30:25+00:00,2016-01-01 05:47:56+00:00,2025-12-31 23:50:42+00:00,19,168,48


,specification_item,frozen_value,evidence_and_rationale
0,Sample split,2016-2024 train; 2025 untouched holdout,"All learned text rules, IDF, thresholds and sc..."
1,Time and wash-in,UTC; first 65 days excluded as focal observations,Lag window needs 35-day gap plus 30-day span; ...
2,Outcome,log1p(funding hours); 72h fast-funding flag as...,"All 1,453,846 rows have raisedDate; 6 negative..."
3,Current choice set,Active same-sector pool primary,All exit timestamps present; compute at postin...
4,Time-based checks,14-day precommitted and 16-day calibrated post...,Training P75=394.782h (16.45d); 14d remains pr...
5,Recent-history pool,"Age [35,65) days; 7-day half-life","Training P95=826.891h (34.45d), rounded up bef..."
6,Text,use primary; description robustness,Both 97.31% nonempty; use is the browse-view f...
7,Narrative representation,Masked hashed unigram+bigram TF-IDF; recurring...,"Rules fit on 2016-2024; no partner ID, so no i..."
8,Pool threshold,Raw n>=10; weighted Kish n>=10; 5/20 sensitivity,Proposal rule retained; small and empty pools ...
9,Model and claim,HDFE with Country + Activity + Week FE; countr...,Conditional association only; Engine scores tr...


,valid_funded_loans,p50_hours,p75_hours,p95_hours
0,1316678,69.245833,394.781875,826.890639


S1 profile and freeze | shapes=[(1, 14), (10, 3), (1, 4)] | mean_null_rates=[0.0, 0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.009


In [3]:
t0 = time.time()
nulls = pd.read_csv(OUTPUTS / "column_null_rates.csv")
status = pd.read_csv(OUTPUTS / "status_raised_crosstab.csv")
pool = pd.read_csv(OUTPUTS / "pool_size_distribution.csv")
display(status)
display(pool)
assert int(profile.loc[0, "negative_durations"]) == 6
print("PASS: six negative-duration records are isolated; all claims use the frozen UTC rules")
checkpoint("S1 quality assertions", nulls, status, pool, started=t0)

,status,loans,raised_date_present,raised_date_missing,valid_raised_event,valid_raised_event_pct
0,funded,1452209,1452209.0,0.0,1452203.0,99.9996
1,refunded,1637,1637.0,0.0,1637.0,100.0000


,period,pool,loans,p01,p25,p50,p75,p99,pct_below_5,pct_below_10,pct_below_20
0,all,active,1453840.0,5.000000,246.000000,758.000000,1242.000000,3198.000000,0.991787,2.027871,4.169578
1,all,posting_14d,1453840.0,31.000000,412.000000,1129.000000,1639.000000,3049.000000,0.124567,0.245419,0.467108
2,all,posting_16d,1453840.0,36.000000,463.000000,1284.000000,1832.000000,3378.000000,0.107852,0.218043,0.384912
3,all,lag_kish,1453840.0,13.230686,483.653699,1450.411581,2036.069134,3663.493271,0.841702,0.955195,1.070063
4,all,lag_completed_kish,1453840.0,13.230686,483.083216,1438.557123,2010.248901,3647.640726,0.842734,0.955195,1.071370
5,train_2016_2024,active,1316678.0,5.000000,241.000000,759.000000,1262.000000,3233.000000,0.992042,2.023274,4.105256
6,train_2016_2024,posting_14d,1316678.0,32.000000,401.000000,1129.000000,1641.000000,3044.000000,0.126455,0.243340,0.433060
7,train_2016_2024,posting_16d,1316678.0,37.000000,451.000000,1285.000000,1838.000000,3413.000000,0.109518,0.219188,0.361972
8,train_2016_2024,lag_kish,1316678.0,12.996494,466.172995,1441.695299,2024.029594,3676.050208,0.849259,0.966523,1.082877
9,train_2016_2024,lag_completed_kish,1316678.0,12.995318,465.483949,1428.370604,1994.302516,3656.620516,0.850398,0.966523,1.084320


PASS: six negative-duration records are isolated; all claims use the frozen UTC rules
S1 quality assertions | shapes=[(28, 3), (2, 6), (15, 11)] | mean_null_rates=[0.0, 0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.009
